# Phase-2 Step 6: Cross-Dataset Generalization

**Objective:** Test how well models generalize across different batteries.

**Experiments:**
1. Train on B0005 → Test on B0006
2. Train on B0005 → Test on B0007
3. Train on B0005 → Test on B0006+B0007
4. Train on B0006 → Test on B0005+B0007
5. Train on B0005+B0006 → Test on B0007

**Expected:** Lower R² than within-battery (this is the hard generalization problem!)

In [ ]:
# --- Portable setup (Windows / macOS / Linux) — do not hardcode paths ---
import importlib.util
from pathlib import Path

def _load_notebook_setup():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "notebooks" / "notebook_setup.py",
        cwd / "notebook_setup.py",
    ]
    if cwd.name == "notebooks":
        candidates.insert(0, cwd / "notebook_setup.py")
    for fp in candidates:
        if fp.is_file():
            spec = importlib.util.spec_from_file_location("_notebook_setup", fp)
            mod = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            return mod
    raise FileNotFoundError(
        "Missing notebooks/notebook_setup.py. Unzip the full project, open a terminal, "
        "cd into the repo root, then run: jupyter notebook"
    )

_ns = _load_notebook_setup()
PROJECT_ROOT = _ns.setup()
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


import sys
import os
import warnings
warnings.filterwarnings('ignore')



import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("✅ Setup complete")

## 1. Load Data

In [ ]:
# Load all features
all_features = pd.read_csv('data/phase2/harmonized/all_features.csv')

print(f"Total samples: {len(all_features)}")
print(f"\nBatteries:")
for bat in all_features['battery_id'].unique():
    bat_df = all_features[all_features['battery_id'] == bat]
    print(f"  {bat}: {len(bat_df)} samples, SOH [{bat_df['soh'].min():.1f}, {bat_df['soh'].max():.1f}]%")

In [ ]:
# Define feature columns
exclude_cols = ['battery_id', 'cycle_number', 'original_index', 'soh', 'has_spikes']
feature_cols = [c for c in all_features.columns if c not in exclude_cols]

print(f"Features: {len(feature_cols)}")

## 2. Define Cross-Battery Experiments

In [ ]:
# Define experiments
EXPERIMENTS = {
    'B0005→B0006': {'train': ['B0005'], 'test': ['B0006']},
    'B0005→B0007': {'train': ['B0005'], 'test': ['B0007']},
    'B0005→B0006+B0007': {'train': ['B0005'], 'test': ['B0006', 'B0007']},
    'B0006→B0005+B0007': {'train': ['B0006'], 'test': ['B0005', 'B0007']},
    'B0007→B0005+B0006': {'train': ['B0007'], 'test': ['B0005', 'B0006']},
    'B0005+B0006→B0007': {'train': ['B0005', 'B0006'], 'test': ['B0007']},
    'B0005+B0007→B0006': {'train': ['B0005', 'B0007'], 'test': ['B0006']},
}

print("Cross-Battery Experiments:")
for name, config in EXPERIMENTS.items():
    print(f"  {name}")

In [ ]:
def normalize_with_train_stats(train_df, test_df, feature_cols):
    """
    Normalize using training statistics only.
    """
    train_norm = train_df.copy()
    test_norm = test_df.copy()
    
    for col in feature_cols:
        mean = train_df[col].mean()
        std = train_df[col].std() + 1e-8
        
        train_norm[col] = (train_df[col] - mean) / std
        test_norm[col] = (test_df[col] - mean) / std
    
    return train_norm, test_norm


def run_cross_battery_experiment(all_features, train_batteries, test_batteries, feature_cols):
    """
    Run a single cross-battery experiment.
    """
    # Split data
    train_df = all_features[all_features['battery_id'].isin(train_batteries)].copy()
    test_df = all_features[all_features['battery_id'].isin(test_batteries)].copy()
    
    # Normalize
    train_norm, test_norm = normalize_with_train_stats(train_df, test_df, feature_cols)
    
    # Prepare data
    X_train = train_norm[feature_cols].values
    y_train = train_norm['soh'].values
    X_test = test_norm[feature_cols].values
    y_test = test_norm['soh'].values
    
    results = {}
    
    # Ridge
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train, y_train)
    y_pred_ridge = ridge.predict(X_test)
    
    results['Ridge'] = {
        'mae': mean_absolute_error(y_test, y_pred_ridge),
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred_ridge)),
        'r2': r2_score(y_test, y_pred_ridge),
        'preds': y_pred_ridge,
        'targets': y_test
    }
    
    # Gradient Boosting
    gb = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
    gb.fit(X_train, y_train)
    y_pred_gb = gb.predict(X_test)
    
    results['GB'] = {
        'mae': mean_absolute_error(y_test, y_pred_gb),
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred_gb)),
        'r2': r2_score(y_test, y_pred_gb),
        'preds': y_pred_gb,
        'targets': y_test
    }
    
    return results, len(train_df), len(test_df)

## 3. Run All Experiments

In [ ]:
print("="*70)
print("CROSS-BATTERY GENERALIZATION RESULTS")
print("="*70)

all_results = {}

for exp_name, config in EXPERIMENTS.items():
    results, n_train, n_test = run_cross_battery_experiment(
        all_features, config['train'], config['test'], feature_cols
    )
    all_results[exp_name] = results
    
    print(f"\n{exp_name} (train: {n_train}, test: {n_test}):")
    print(f"  Ridge: MAE={results['Ridge']['mae']:.4f}, RMSE={results['Ridge']['rmse']:.4f}, R²={results['Ridge']['r2']:.4f}")
    print(f"  GB:    MAE={results['GB']['mae']:.4f}, RMSE={results['GB']['rmse']:.4f}, R²={results['GB']['r2']:.4f}")

## 4. Visualize Results

In [ ]:
# Create summary DataFrame
summary_data = []
for exp_name, results in all_results.items():
    for model_name, metrics in results.items():
        summary_data.append({
            'Experiment': exp_name,
            'Model': model_name,
            'MAE': metrics['mae'],
            'RMSE': metrics['rmse'],
            'R²': metrics['r2']
        })

summary_df = pd.DataFrame(summary_data)
print("\nSummary:")
print(summary_df.to_string(index=False))

In [ ]:
# Visualize R² by experiment
fig, ax = plt.subplots(figsize=(12, 6))

experiments = list(all_results.keys())
x = np.arange(len(experiments))
width = 0.35

ridge_r2 = [all_results[e]['Ridge']['r2'] for e in experiments]
gb_r2 = [all_results[e]['GB']['r2'] for e in experiments]

bars1 = ax.bar(x - width/2, ridge_r2, width, label='Ridge', color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, gb_r2, width, label='Gradient Boosting', color='forestgreen', edgecolor='black')

ax.axhline(y=0, color='black', linewidth=1)
ax.axhline(y=0.9, color='green', linestyle='--', alpha=0.5, label='Within-battery baseline (~0.9)')

ax.set_ylabel('R²', fontsize=12)
ax.set_xlabel('Experiment', fontsize=12)
ax.set_title('Cross-Battery Generalization: R² Scores', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(experiments, rotation=45, ha='right', fontsize=10)
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bar, val in zip(bars1, ridge_r2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{val:.2f}', ha='center', fontsize=8)
for bar, val in zip(bars2, gb_r2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('results/phase2/figures/step6_cross_battery_r2.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Saved: results/phase2/figures/step6_cross_battery_r2.png")

In [ ]:
# Prediction plots for key experiments
key_experiments = ['B0005→B0006', 'B0005→B0007', 'B0005+B0006→B0007']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, exp_name in zip(axes, key_experiments):
    results = all_results[exp_name]['Ridge']
    
    ax.scatter(results['targets'], results['preds'], alpha=0.6, s=30)
    
    min_val = min(results['targets'].min(), results['preds'].min())
    max_val = max(results['targets'].max(), results['preds'].max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
    
    ax.set_xlabel('Actual SOH (%)')
    ax.set_ylabel('Predicted SOH (%)')
    ax.set_title(f'{exp_name}\nR² = {results["r2"]:.4f}', fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Cross-Battery Predictions (Ridge)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/phase2/figures/step6_cross_battery_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Saved: results/phase2/figures/step6_cross_battery_predictions.png")

## 5. Analysis

In [ ]:
print("\n" + "="*70)
print("CROSS-BATTERY GENERALIZATION ANALYSIS")
print("="*70)

# Within-battery baseline (from previous step)
within_battery_r2 = 0.9176  # Ridge result from Step 3

print(f"\n📊 Within-battery baseline R²: {within_battery_r2:.4f}")
print("\n📉 Cross-battery R² (Ridge):")

for exp_name in experiments:
    r2 = all_results[exp_name]['Ridge']['r2']
    drop = within_battery_r2 - r2
    drop_pct = (drop / within_battery_r2) * 100
    
    status = "🟢" if r2 > 0.5 else "🟡" if r2 > 0 else "🔴"
    print(f"  {status} {exp_name}: R²={r2:.4f} (↓{drop_pct:.1f}% from within-battery)")

In [ ]:
# Identify best and worst generalization
ridge_r2_dict = {e: all_results[e]['Ridge']['r2'] for e in experiments}

best_exp = max(ridge_r2_dict.items(), key=lambda x: x[1])
worst_exp = min(ridge_r2_dict.items(), key=lambda x: x[1])

print(f"\n🏆 Best generalization: {best_exp[0]} (R²={best_exp[1]:.4f})")
print(f"📉 Worst generalization: {worst_exp[0]} (R²={worst_exp[1]:.4f})")

## 6. Save Results

In [ ]:
# Save results
summary_df.to_csv('results/phase2/step6_cross_battery_results.csv', index=False)
print("✅ Saved: results/phase2/step6_cross_battery_results.csv")

In [ ]:
print("\n" + "="*70)
print("STEP 6: SUMMARY")
print("="*70)
print("""
KEY FINDINGS:

1. Cross-battery generalization is MUCH harder than within-battery prediction
   - Within-battery R² ≈ 0.92
   - Cross-battery R² varies widely (-0.2 to 0.7)

2. Training on more batteries helps generalization
   - Single battery training → poor generalization
   - Multi-battery training → better generalization

3. Some battery pairs generalize better than others
   - This reflects different degradation patterns/conditions

4. Negative R² is expected and scientifically valid
   - It means model is worse than predicting test mean
   - This is the "domain shift" problem in ML

IMPLICATIONS FOR RESEARCH:

- Real-world deployment needs multi-battery training data
- Domain adaptation techniques could help
- Physics-based features may transfer better than raw data
- This justifies the need for your multimodal approach!
""")

print("\n✅ Step 6 Complete - Proceed to Step 7: Robustness Testing")